# Virginia Climate Center - Energy Assistance Programming and Climate Relationships

This notebook is a starting point for analysis in this repository. The Python project configuration and `.venv` are at the repository root.

In [16]:
import sys

import matplotlib
import numpy
import pandas as pd
import scipy
import seaborn
from pathlib import Path

### Load and process HDD and CDD data

Each file has a 10-character identifier followed by January–December values. The identifier contains the area code (3 digits), division number (1 digit), element code (2 digits), and year (4 digits). For example, `0010261895` means area `001`, division `0`, CDD element `26`, year `1895`; HDD uses element `25`. The cell below reads both files and splits the identifier into separate columns.

Virginia is in area `044`

In [17]:
data_dir = Path("data") if Path("data").is_dir() else Path("../data")
months = ["january", "february", "march", "april", "may", "june",
          "july", "august", "september", "october", "november", "december"]
columns = ["record_id", *months]
widths = [10, *([7] * 12)]

hdd_df = pd.read_fwf(
    data_dir / "climdiv-hddcst-v1.0.0-20261006.txt",
    widths=widths,
    names=columns,
    dtype={"record_id": "string"},
    na_values=["-9999."],
)
cdd_df = pd.read_fwf(
    data_dir / "climdiv-cddcst-v1.0.0-20261006.txt",
    widths=widths,
    names=columns,
    dtype={"record_id": "string"},
    na_values=["-9999."],
)

for df in (hdd_df, cdd_df):
    df["area_code"] = df["record_id"].str[:3]
    df["division_number"] = df["record_id"].str[3:4].astype(int)
    df["element_code"] = df["record_id"].str[4:6]
    df["year"] = df["record_id"].str[6:10].astype(int)

display(hdd_df.head())
display(cdd_df.head())

,record_id,january,february,march,april,may,june,july,august,september,october,november,december,area_code,division_number,element_code,year
0,0010251895,717.0,800.0,373.0,115.0,34.0,0.0,0.0,0.0,0.0,234.0,377.0,646.0,001,0,25,1895
1,0010251896,693.0,517.0,429.0,47.0,4.0,0.0,0.0,0.0,4.0,147.0,275.0,603.0,001,0,25,1896
2,0010251897,752.0,422.0,239.0,135.0,39.0,0.0,0.0,0.0,2.0,75.0,355.0,573.0,001,0,25,1897
3,0010251898,545.0,556.0,246.0,229.0,8.0,0.0,0.0,0.0,3.0,181.0,478.0,694.0,001,0,25,1898
4,0010251899,690.0,732.0,342.0,148.0,3.0,0.0,0.0,0.0,10.0,87.0,321.0,661.0,001,0,25,1899


,record_id,january,february,march,april,may,june,july,august,september,october,november,december,area_code,division_number,element_code,year
0,0010261895,5.0,0.0,18.0,49.0,155.0,359.0,419.0,437.0,365.0,28.0,4.0,1.0,001,0,26,1895
1,0010261896,4.0,3.0,12.0,118.0,320.0,354.0,488.0,522.0,305.0,60.0,20.0,2.0,001,0,26,1896
2,0010261897,3.0,10.0,55.0,38.0,143.0,464.0,496.0,414.0,309.0,121.0,8.0,3.0,001,0,26,1897
3,0010261898,19.0,2.0,45.0,13.0,268.0,446.0,457.0,421.0,298.0,37.0,1.0,1.0,001,0,26,1898
4,0010261899,5.0,0.0,24.0,32.0,331.0,436.0,466.0,490.0,225.0,105.0,12.0,2.0,001,0,26,1899


## Filter for Virginia data

In [18]:
va_hdd_df = hdd_df[hdd_df["area_code"]== "044"]
va_cdd_df = cdd_df[cdd_df["area_code"]== "044"]
display(va_hdd_df.head())
display(va_cdd_df.head())


,record_id,january,february,march,april,may,june,july,august,september,october,november,december,area_code,division_number,element_code,year
5676,0440251895,1047.0,1108.0,701.0,355.0,142.0,2.0,4.0,0.0,11.0,433.0,574.0,854.0,044,0,25,1895
5677,0440251896,983.0,790.0,784.0,246.0,35.0,9.0,0.0,0.0,46.0,364.0,429.0,911.0,044,0,25,1896
5678,0440251897,1042.0,763.0,561.0,341.0,143.0,7.0,0.0,2.0,36.0,240.0,558.0,827.0,044,0,25,1897
5679,0440251898,845.0,836.0,492.0,443.0,85.0,3.0,0.0,0.0,22.0,245.0,648.0,899.0,044,0,25,1898
5680,0440251899,954.0,1020.0,664.0,338.0,85.0,1.0,0.0,0.0,55.0,249.0,576.0,899.0,044,0,25,1899


,record_id,january,february,march,april,may,june,july,august,september,october,november,december,area_code,division_number,element_code,year
5676,0440261895,0.0,0.0,0.0,0.0,36.0,251.0,231.0,312.0,209.0,1.0,0.0,0.0,044,0,26,1895
5677,0440261896,0.0,0.0,0.0,6.0,153.0,186.0,335.0,301.0,105.0,4.0,0.0,0.0,044,0,26,1896
5678,0440261897,0.0,0.0,1.0,0.0,41.0,191.0,339.0,261.0,127.0,19.0,0.0,0.0,044,0,26,1897
5679,0440261898,0.0,0.0,5.0,0.0,78.0,226.0,374.0,338.0,162.0,18.0,0.0,0.0,044,0,26,1898
5680,0440261899,0.0,0.0,0.0,0.0,79.0,278.0,338.0,326.0,94.0,17.0,0.0,0.0,044,0,26,1899


### Virginia Energy Assistance Programming Notes
Virginia's Energy Assistance Program (EAP) supports low income households meet their home energy needs. EAP is administered by the Virginia Department of Social Services, and has four parts:

- Fuel assistance: offsets heating fuel costs
  - One time heating benefit toward heating costs, can cover fuel, delivery, late charges, and connection or restart fees. Amount is set by a point system (income, heat type, climate zone, vulnerability). Reapply every year.
  - Application window is mid October to mid November, and payments start in January
- Crisis assistance: helps with heating emergencies
  - Crisis: equipment, deposits, shelter
    - Heating equipment repair or replacement, space heaters, one-time heat security deposit, emergency shelter
    - Nov 1 to Mar 15, payments start immediately upon application approval
  - Crisis: fuel and utility bills
    - Purchase of primary heating fuel and payment of heat utility bills
    - First workday of January to March 15, payments start immediately upon application approval
- Cooling assistance: helps with cooling bills and equipment
  - One-time electric bill payment, electric security deposit, central AC or heat pump repair, whole-house fan, window unit
  - June 15 to August 15
  - One time benefit, payment upon application approval
  - Household needs a person 60 or older, a person with a disability, or a child under 6. First-come, first-served, only when federal funds allow. 
- Weatherization assistance: improves energy efficiency and reduces costs
  - Not relevant to seasonal changes, pertains more to energy equipment upgrades

To qualify, you must have heating/cooling expenses and have a gross monthly income that is 150% of the federal poverty level.

LIHEAP was established shortly after the 1970s oil crisis to primarily help residences pay for heating. As of 2024, LIHEAP allocates over half its funding to heating assistance, with less than 10 % supporting cooling assistance despite increasing demand for air conditioning amid rising temperatures due to climate change


### Main analysis questions
- How have monthly HDD and CDD changed across periods? And how have they changed since LIHEAP was introduced?
- Is cooling demand spreading outside June to August? And by how much?
- Does the heating window line up with the heating need? The fuel application window (mid-October to mid-November) comes before most demand, and payments arrive in January. When do HDD actually peak, and has that shifted?
- Does the crisis window cover the cold months? Crisis runs through March 15. How much heating demand falls after that date, in late March and April?
- Do regions differ? Could repeat the analysis by climate division (Virginia has several). The regulation uses six NOAA zones for heating benefits.